# Combined ML Training, Testing, and Visuals

This Colab uses the ByteSmart cooling dataset, checks whether the selected sample covers the larger dataset, splits the sample into train/test data, trains Linear Regression, K-Means, and Logistic Regression, evaluates them, and plots visuals for all three algorithms.

In [ ]:
import os
import glob
import io
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42
MODEL_DIR = Path("/content/ml_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DATASET_FILE_ID = "1sEynbX9BezEIS4MBdbY2jWsgkii4uJON"
DRIVE_MODEL_FOLDER_ID = "1elG38b6F7Ptfq2spqnW5zAxzqgPR4vDh"

In [ ]:
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

expected_paths = [
    "/content/drive/MyDrive/ByteSmart Internship - Ranveer Singh/cold_source_control_dataset.csv",
]

csv_path = None
for candidate in expected_paths:
    if os.path.exists(candidate):
        csv_path = candidate
        break

if csv_path is None:
    matches = glob.glob(
        "/content/drive/**/ByteSmart Internship - Ranveer Singh/cold_source_control_dataset.csv",
        recursive=True,
    )
    if not matches:
        print("Could not find the CSV in mounted Drive paths. Downloading it by Drive file ID instead.")
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload

        auth.authenticate_user()
        drive_service = build("drive", "v3")
        request = drive_service.files().get_media(fileId=DRIVE_DATASET_FILE_ID)
        csv_path = "/content/cold_source_control_dataset.csv"
        with open(csv_path, "wb") as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Download progress: {int(status.progress() * 100)}%")
    else:
        csv_path = matches[0]

df = pd.read_csv(csv_path)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

rename_map = {
    "Inlet_Temperature(°C)": "Inlet_Temperature",
    "Outlet_Temperature(°C)": "Outlet_Temperature",
    "Ambient_Temperature(°C)": "Ambient_Temperature",
    "Cooling_Unit_Power_Consumption(kW)": "Cooling_Power_kW",
    "Chiller_Usage(%)": "Chiller_Usage",
    "AHU_Usage(%)": "AHU_Usage",
    "Total_Energy_Cost($)": "Total_Energy_Cost",
    "Temperature_Deviation(°C)": "Temperature_Deviation",
    "Server_Workload(%)": "Server_Workload",
}
df = df.rename(columns=rename_map)
df["Thermal_Delta"] = df["Outlet_Temperature"] - df["Inlet_Temperature"]
df["Increase_Chiller_Label"] = (df["Cooling_Strategy_Action"] == "Increase Chiller").astype(int)

feature_cols = [
    "Server_Workload",
    "Inlet_Temperature",
    "Outlet_Temperature",
    "Ambient_Temperature",
    "Chiller_Usage",
    "AHU_Usage",
    "Temperature_Deviation",
    "Thermal_Delta",
]
regression_target = "Cooling_Power_kW"
classification_target = "Increase_Chiller_Label"
cluster_cols = feature_cols + ["Cooling_Power_kW", "Total_Energy_Cost"]

print("Dataset:", csv_path)
print("Shape:", df.shape)
display(df.head())

## Sample Coverage Check

In [ ]:
sample_df, remainder_df = train_test_split(
    df,
    train_size=0.80,
    random_state=RANDOM_STATE,
    stratify=df["Cooling_Strategy_Action"],
)

coverage_rows = []
numeric_cols = [
    "Server_Workload",
    "Inlet_Temperature",
    "Outlet_Temperature",
    "Ambient_Temperature",
    "Cooling_Power_kW",
    "Chiller_Usage",
    "AHU_Usage",
    "Total_Energy_Cost",
    "Temperature_Deviation",
]

for col in numeric_cols:
    coverage_rows.append({
        "column": col,
        "full_min": df[col].min(),
        "sample_min": sample_df[col].min(),
        "full_max": df[col].max(),
        "sample_max": sample_df[col].max(),
        "full_mean": df[col].mean(),
        "sample_mean": sample_df[col].mean(),
        "mean_difference": sample_df[col].mean() - df[col].mean(),
    })

coverage = pd.DataFrame(coverage_rows)
strategy_coverage = pd.concat(
    [
        df["Cooling_Strategy_Action"].value_counts(normalize=True).rename("full_pct"),
        sample_df["Cooling_Strategy_Action"].value_counts(normalize=True).rename("sample_pct"),
    ],
    axis=1,
).fillna(0)
strategy_coverage["pct_point_difference"] = (
    strategy_coverage["sample_pct"] - strategy_coverage["full_pct"]
) * 100

display(coverage.round(3))
display(strategy_coverage.round(3))

## Train/Test Split

In [ ]:
train_df, test_df = train_test_split(
    sample_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=sample_df["Cooling_Strategy_Action"],
)

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_reg_train = train_df[regression_target]
y_reg_test = test_df[regression_target]

y_log_train = train_df[classification_target]
y_log_test = test_df[classification_target]

X_cluster_train = train_df[cluster_cols]
X_cluster_test = test_df[cluster_cols]

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Test percentage of selected sample:", round(len(test_df) / len(sample_df) * 100, 2), "%")

## Train the Three Algorithms

In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
linear_model.fit(X_train, y_reg_train)

kmeans_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)),
])
kmeans_model.fit(X_cluster_train)

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
logistic_model.fit(X_train, y_log_train)

print("Models trained: Linear Regression, K-Means, Logistic Regression")

## Test and Evaluate

In [ ]:
linear_pred = linear_model.predict(X_test)
linear_metrics = {
    "r2": r2_score(y_reg_test, linear_pred),
    "mae": mean_absolute_error(y_reg_test, linear_pred),
    "rmse": np.sqrt(mean_squared_error(y_reg_test, linear_pred)),
}
print("Linear Regression metrics")
for key, value in linear_metrics.items():
    print(f"{key}: {value:.4f}")

kmeans_labels = kmeans_model.predict(X_cluster_test)
kmeans_silhouette = silhouette_score(
    kmeans_model.named_steps["scaler"].transform(X_cluster_test),
    kmeans_labels,
)
print(f"\nK-Means silhouette score on test data: {kmeans_silhouette:.4f}")

logistic_pred = logistic_model.predict(X_test)
logistic_proba = logistic_model.predict_proba(X_test)[:, 1]
logistic_accuracy = accuracy_score(y_log_test, logistic_pred)
print(f"\nLogistic Regression accuracy: {logistic_accuracy:.4f}")
print(classification_report(y_log_test, logistic_pred, target_names=["Other action", "Increase Chiller"]))

## Visuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_reg_test, linear_pred, alpha=0.65)
axes[0].plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], "r--")
axes[0].set_title("Linear Regression: Actual vs Predicted Power")
axes[0].set_xlabel("Actual Cooling Power (kW)")
axes[0].set_ylabel("Predicted Cooling Power (kW)")

cluster_plot = axes[1].scatter(
    X_cluster_test["Server_Workload"],
    X_cluster_test["Cooling_Power_kW"],
    c=kmeans_labels,
    cmap="viridis",
    alpha=0.75,
)
axes[1].set_title("K-Means: Test Data Operating Clusters")
axes[1].set_xlabel("Server Workload (%)")
axes[1].set_ylabel("Cooling Power (kW)")
fig.colorbar(cluster_plot, ax=axes[1], label="Cluster")

axes[2].scatter(X_test["Chiller_Usage"], logistic_proba, c=y_log_test, cmap="coolwarm", alpha=0.75)
axes[2].axhline(0.5, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Logistic Regression: Increase Chiller Probability")
axes[2].set_xlabel("Chiller Usage (%)")
axes[2].set_ylabel("Predicted Probability")

plt.tight_layout()
plt.show()

cm = confusion_matrix(y_log_test, logistic_pred)
ConfusionMatrixDisplay(cm, display_labels=["Other action", "Increase Chiller"]).plot(cmap="Blues", values_format="d")
plt.title("Logistic Regression Confusion Matrix")
plt.show()

## Interpretation Notes

Linear Regression is strong if R2 is close to 1 and MAE/RMSE are small compared with the cooling-power range.

K-Means is useful if the silhouette score is meaningfully above 0 and the clusters separate operating conditions in a way that makes engineering sense.

Logistic Regression is strong if accuracy, precision, and recall are high for the `Increase Chiller` class, not just overall accuracy.